# ⚽ Mission 12: Soccer Vision Lab — Build an AI Tactical Analyst

## 📖 Mission Story
Welcome back! 🎉

During the first eleven missions, you gradually learned how to become an AI-powered soccer coach. You started with Python programming, then learned how to organize data with Pandas, visualize player statistics, build professional Streamlit dashboards, and even consult Gemini AI to answer tactical questions.

But there has always been one important limitation: every dashboard you built assumed that someone had already collected the player statistics (e.g., Artin_FC_v2 dataset). Professional soccer clubs don't work that way. Instead of receiving ready-made numbers, they begin with a match video!

Today, you are going to build your own sports analytics system. By the end of this mission, your Soccer AI Coach will no longer depend on manually entered statistics—it will create its own statistics directly from match footage. Welcome to the world of **Computer Vision**!

---

## 🎯 Learning Objectives
* ✅ Explain how computers represent videos as sequences of digital images (frames).
* ✅ Read soccer videos frame-by-frame using OpenCV (`cv2`).
* ✅ Upload video files inside Streamlit using `st.file_uploader()`.
* ✅ Detect soccer players inside video frames using YOLO (`ultralytics`).
* ✅ Understand the key difference between **Object Detection** and **Object Tracking**.
* ✅ Convert camera pixel coordinates into real soccer pitch coordinates ($120 \times 80$ yards).
* ✅ Generate professional tactical heatmaps using `mplsoccer`.
* ✅ Ask Gemini AI to analyze player movement as an elite UEFA Pro Analyst.
* ✅ Combine Computer Vision, Data Science, Visualization, and AI into one complete application.

---

## 🟢 Phase 1: Understanding Soccer Heatmaps

Before we teach the computer how to collect movement from video, we first need to learn how to visualize movement on a soccer pitch.

In modern sports analytics, a **Heatmap** shows where a player spends most of their time during a match. Every time a player touches the ball or moves into space, we record their position as an $(X, Y)$ coordinate:
- **X Coordinate (0 to 120 yards):** The length of the pitch (0 = own goal line, 120 = opponent's goal line).
- **Y Coordinate (0 to 80 yards):** The width of the pitch (0 = left touchline, 80 = right touchline).

Let's write a program using `mplsoccer` and `matplotlib` to plot pitch movement:

### 🧰 New Libraries

Today we introduce:

| Library     | Purpose                           |
| ----------- | --------------------------------- |
| mplsoccer   | Professional soccer visualization |
| OpenCV      | Video processing                  |
| YOLO        | Player detection                  |
| Ultralytics | Computer vision models            |
| Gemini API  | Tactical intelligence             |


### Step 1. Your First Soccer Analytics Pitch

Before analyzing videos, we need to understand: How does a computer represent a player's movement?

A human sees:

"Artin moved from defense to attack."

A computer sees coordinates:

**Exercise 1.1:** Draw Artin's Movement

Run:

In [11]:
!pip install mplsoccer 


ERROR: Exception:
Traceback (most recent call last):
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/commands/install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/metadata/pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "/Users/maryamraiyataliabadi/opt/anacond

In [10]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch


pitch = Pitch(
    pitch_type="statsbomb"
)


fig, ax = pitch.draw(figsize=(10,7))


x = [10,20,30,45,60,75,90]

y = [60,55,50,40,30,25,20]


pitch.scatter(
    x,
    y,
    ax=ax,
    s=80,
    color="black"
)


plt.title(
"Artin Movement Map"
)


plt.show()

ImportError: cannot import name 'colormaps' from 'matplotlib' (/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/matplotlib/__init__.py)

In [12]:
import matplotlib

print(matplotlib.__version__)

3.3.4


In [13]:
!pip install --upgrade matplotlib

ERROR: Exception:
Traceback (most recent call last):
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/commands/install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "/Users/maryamraiyataliabadi/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/metadata/pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "/Users/maryamraiyataliabadi/opt/anacond

### Step2. Creating Real Player Movement Data

Professional systems do not store movement as Python lists.

They store tables.

Example:

| frame | x  | y  |
| ----- | -- | -- |
| 1     | 10 | 60 |
| 2     | 20 | 55 |
| 3     | 30 | 50 |


**Exercise 1.2:** Create Movement DataFrame

Run:

In [5]:
import pandas as pd


movement = {

"frame":[1,2,3,4,5],

"x":[10,20,35,50,70],

"y":[60,55,45,35,25]

}


df = pd.DataFrame(movement)


df

,frame,x,y
0,1,10,60
1,2,20,55
2,3,35,45
3,4,50,35
4,5,70,25


### Step3.  Generate a Player Heatmap

Now we answer:

Where does Artin spend most of his time?

In [6]:
from mplsoccer import Pitch
import matplotlib.pyplot as plt


pitch = Pitch(
pitch_type="statsbomb"
)


fig,ax=pitch.draw()


pitch.kdeplot(
df["x"],
df["y"],
ax=ax,
fill=True
)


pitch.scatter(
df["x"],
df["y"],
ax=ax
)


plt.title(
"Artin Movement Heatmap"
)


plt.show()

ModuleNotFoundError: No module named 'mplsoccer'

**Exercise 1.3:**

Change the coordinates.

Create:

Version A: Attacking winger

Version B: Defensive fullback

Compare the heatmaps.

### Step4. Streamlit Soccer Vision App

Now we convert our notebook into a professional application.

Remember Mission 9:

Create a Streamlit app named: `Soccer_Vision_App.py`

In [ ]:
import streamlit as st


st.title(
"⚽ Artin FC Soccer Vision Lab"
)


st.write(
"""
AI-powered soccer movement analysis system
"""
)

Now add select the player and position:

In [ ]:
player_name = st.text_input("Player Name")

position = st.selectbox(
"Position",
[
"Forward",
"Midfielder",
"Defender"
]
)

### Step 5. Upload Soccer Video
Now your app can receive match footage.

In [ ]:
video = st.file_uploader(
"Upload Soccer Video",
type=["mp4"]
)

# if uploaded
if video:

    st.video(video)

**Exercise 1.4:**

Upload a short soccer clip.

Answer:

- What is the video resolution?
- How many frames does it contain? 

### Step 6. Understanding Video with OpenCV

A video is simply:

`Frame 1
 Frame 2
 Frame 3
 ...
 Frame 100`

OpenCV allows us to read these frames.

In [ ]:
import cv2


cap=cv2.VideoCapture("video.mp4")


frames=int(
cap.get(cv2.CAP_PROP_FRAME_COUNT)
)


fps=cap.get(
cv2.CAP_PROP_FPS
)


print(frames)
print(fps)

**Exercise 1.5:**
Create a Streamlit metric panel:

In [ ]:
st.metric(
"Total Frames",
frames
)

### Step 7.Introducing AI Player Tracking

Professional systems cannot manually click players. They use computer vision.

Today we use: `YOLO` (You Only Look Once)

`YOLO` can detect:

- players
- balls
- referees
- objects

Install `pip install ultralytics`

Load model:

In [ ]:

from ultralytics import YOLO

model=YOLO(
"yolov8n.pt"
)

Analyze frame:


In [ ]:

results=model(frame)

The AI returns:

Print the detection results.

In [ ]:
print(results[0])

### Step 8. Convert Video Coordinates to Soccer Coordinates

A camera sees:

1280 x 720 pixels

Soccer uses:

120 x 80 yards

We need conversion.

In [ ]:
def convert_coordinates(
pixel_x,
pixel_y
):


    soccer_x = (
    pixel_x/1280
    )*120


    soccer_y = (
    pixel_y/720
    )*80


    return soccer_x,soccer_y

### Step 9. Build Tactical Intelligence
Now we combine everything with Gemini.

Remember Mission 10:

AI needs a good prompt.

Now we combine everything with Gemini.

Remember Mission 10:

AI needs a good prompt.

In [ ]:
summary=f"""

Player:
Artin


Position:
Forward


Average position:
{x.mean()}, {y.mean()}


Movement points:
{len(df)}

"""

In [ ]:
prompt=f"""

You are a UEFA Pro soccer analyst.

Analyze this player.

{summary}


Provide:

1. Strengths

2. Weaknesses

3. Training recommendations

4. Tactical advice


"""

prompt=f"""

You are a UEFA Pro soccer analyst.

Analyze this player.

{summary}


Provide:

1. Strengths

2. Weaknesses

3. Training recommendations

4. Tactical advice


"""

## Final Boss Challenge 🏆

Build:
Artin FC AI Tactical Center

Your app must include:

1. Player Selection

2. Position Selection

3. Movement Dashboard

Display:

Distance Covered

Average Position

Most Frequent Zone

4. Visualization

Generate:

movement map
heatmap
pitch visualization

5. AI Tactical Report

Gemini should answer:

Example:

"Artin is a right winger. His movement shows frequent attacks into the final third but limited defensive recovery. Recommend improving transition speed."

### ⭐ Super Challenge

Add:

Upload a Real Match Video

The system automatically:

- Detects players
- Tracks movement
- Generates heatmap
- Creates AI coaching report

In [7]:
import matplotlib.pyplot as plt
import mplsoccer
from mplsoccer import Pitch
import numpy as np

# 1. Create a professional soccer pitch layout
pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

# 2. Simulate positional data
player = "Artin"
position = "Right Defender"

if position == "Left Winger":
    x_coordinates = [10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105]
    y_coordinates = [10,12,15,18,20,18,15,12,10,15,18,20,22,25,28,30,35,38,40,42]
elif position == "Striker":
    x_coordinates = [70,75,80,85,90,95,100,102,105,108,110,112,115,108,104,100,95,90,88,110]
    y_coordinates = [35,38,40,42,40,38,35,37,40,42,39,36,40,45,48,50,45,42,38,35]
elif position == "Midfielder":
    x_coordinates = [35,40,45,50,55,60,65,70,60,55,50,45,40,55,65,75,70,60,50,45]
    y_coordinates = [25,30,35,40,45,40,35,30,25,20,25,30,35,45,50,45,40,35,30,25]
elif position == "Right Defender":
    x_coordinates = [15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110]
    y_coordinates = [70,72,68,70,72,74,76,74,72,70,68,70,72,74,76,72,68,65,60,55]

# 3. Plot the touches as a heatmap (2D Histogram)
bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

# 4. Draw individual touch points on top
pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

plt.title(f"{player}'s Soccer Heatmap - {position}", fontsize=18, fontweight='bold', pad=15)
plt.show()

ModuleNotFoundError: No module named 'mplsoccer'

### ✏️ Phase 1 Practice Exercises

**Exercise 1.1:** Change the `position` variable to `"Left Winger"` and re-run the code above. Observe how the heatmap concentration shifts to the top sideline.

**Exercise 1.2:** Modify the colormap in `pitch.heatmap()` from `'Reds'` to `'plasma'` or `'Blues'` and note how it changes the visual contrast.

## 📊 Phase 2: From Manual Coordinates to Real Player Data

In real applications, coordinates are loaded dynamically from files. We manage movement tracking logs using **Pandas DataFrames** and store them into `.csv` files.

In [ ]:
%%writefile phase2_csv_loader.py
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Create synthetic player tracking dataset
data = {
    'frame': list(range(1, 11)),
    'x': [18, 20, 21, 25, 30, 42, 55, 68, 72, 85],
    'y': [52, 53, 54, 50, 48, 45, 40, 38, 35, 30]
}

# Save to CSV
df = pd.DataFrame(data)
df.to_csv("player_movement.csv", index=False)
print("✅ CSV file 'player_movement.csv' created successfully!")

# Load CSV and render Kernel Density Estimate (KDE) smooth heatmap
movement_df = pd.read_csv("player_movement.csv")
print("Loaded Data:")
print(movement_df.head())

pitch = Pitch(pitch_type='statsbomb', pitch_color='#101010', line_color='#888888')
fig, ax = pitch.draw(figsize=(10, 7))
pitch.kdeplot(movement_df['x'], movement_df['y'], ax=ax, cmap='magma', fill=True, alpha=0.65)
plt.title("Artin Movement Trajectory (KDE Density)")
plt.savefig("phase2_heatmap.png", bbox_inches='tight')
print("✅ Saved plot as 'phase2_heatmap.png'!")

### ✏️ Phase 2 Practice Exercises

**Exercise 2.1:** Execute `python phase2_csv_loader.py` in your terminal and inspect the output image `phase2_heatmap.png`.

**Exercise 2.2:** Add 5 additional frames to the synthetic dataset showing the player moving back toward own goal line (lowering $X$ values) and re-generate the CSV.

## 🎥 Phase 3: Reading Soccer Videos with OpenCV

A video is simply a fast sequence of still images called **frames**. OpenCV (`cv2`) allows us to:
1. Open a video file using `cv2.VideoCapture()`.
2. Extract technical parameters: **Resolution** (Width $\times$ Height), **FPS** (Frames Per Second), and **Total Frame Count**.
3. Read individual frames as NumPy arrays for image processing.

In [ ]:
%%writefile phase3_opencv_intro.py
import cv2

def inspect_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open video file: {video_path}")
        return
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    print("📹 Video Metadata Summary")
    print("------------------------")
    print(f"Resolution : {width} x {height} pixels")
    print(f"FPS        : {fps:.2f} frames/sec")
    print(f"Frames     : {frame_count}")
    print(f"Duration   : {duration:.2f} seconds")
    
    # Save first frame to disk
    success, frame = cap.read()
    if success:
        cv2.imwrite("first_frame.jpg", frame)
        print("✅ Saved first frame as 'first_frame.jpg'")
    
    cap.release()

# Test execution guard (uncomment when video file is present)
# inspect_video("sample_match.mp4")

### ✏️ Phase 3 Practice Exercises

**Exercise 3.1:** What is the duration in seconds of a video recorded at 30 FPS with a total frame count of 900? Write down your answer.

**Exercise 3.2:** Modify the code to extract and save the 50th frame of a video clip instead of the first frame.

## 💻 Phase 4: Building a Soccer Video Upload Interface with Streamlit

To process uploaded videos dynamically in a web app, we combine Streamlit's `st.file_uploader()` with Python's built-in `tempfile` module. OpenCV needs a physical file path to read video frames.

In [ ]:
%%writefile phase4_video_upload.py
import streamlit as st
import cv2
import tempfile

st.title("⚽ Soccer Video Analyzer")
uploaded_file = st.file_uploader("Upload Match Video", type=["mp4", "mov", "avi"])

if uploaded_file is not None:
    st.video(uploaded_file)
    
    # Write stream buffer into a temporary disk file
    tfile = tempfile.NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    video_path = tfile.name
    
    # Inspect file via OpenCV
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    st.subheader("📹 Technical Video Metrics")
    st.write(f"**Resolution:** {width} x {height} px")
    st.write(f"**FPS:** {fps:.2f}")
    st.write(f"**Total Frames:** {frame_count}")
    if fps > 0:
        st.write(f"**Duration:** {frame_count / fps:.2f} seconds")
    cap.release()

### ✏️ Phase 4 Practice Exercises

**Exercise 4.1:** Launch this interface using `streamlit run phase4_video_upload.py` and test uploading a short match clip.

**Exercise 4.2:** Why do we need `tempfile.NamedTemporaryFile` instead of passing `uploaded_file` directly into `cv2.VideoCapture()`? Explain in your own words.

## 🤖 Phase 5: YOLO — Detection & Object Tracking

### Object Detection vs. Object Tracking
- **Object Detection:** Identifies *what* is in an image frame (e.g., "Person at $[x1, y1, x2, y2]$"). In every frame, it treats detections independently.
- **Object Tracking:** Assigns a persistent **ID number** (Track ID) to each specific individual so they can be followed across consecutive video frames over time.

We use the **YOLOv8** model from `ultralytics` to perform real-time tracking.

In [ ]:
%%writefile phase5_yolo_demo.py
from ultralytics import YOLO

# Load lightweight YOLOv8 nano model
model = YOLO("yolov8n.pt")
print("✅ Pretrained YOLOv8 model loaded successfully!")

# Tracking invocation pattern:
# results = model.track(source="sample_match.mp4", persist=True)
# 'persist=True' instructs tracker to retain Track IDs between consecutive frames.

### ✏️ Phase 5 Practice Exercises

**Exercise 5.1:** What class index corresponds to a human / player in COCO-trained YOLO models? (Hint: Class index `0`).

**Exercise 5.2:** Explain why tracking requires `persist=True` when processing video frames sequential loop.

## 🟢 Phase 6: Coordinate Mapping — From Pixel Space to Pitch Space

Camera coordinates are measured in **Pixels** ($1280 \times 720$). Tactical maps require dimensions measured in **Yards** ($120 \times 80$).

To transform coordinates, we calculate proportional scaling and apply bounding constraints (`clamp`):

In [ ]:
%%writefile tracking_utils.py
def clamp(value, minimum, maximum):
    """Restricts a numeric value within strict bounds."""
    return max(minimum, min(value, maximum))

def map_pixels_to_pitch(pixel_x, pixel_y, frame_width=1280, frame_height=720, pitch_length=120.0, pitch_width=80.0):
    """
    Transforms pixel screen coordinates (X, Y) into StatsBomb pitch coordinates (0-120, 0-80).
    """
    raw_pitch_x = (pixel_x / frame_width) * pitch_length
    raw_pitch_y = (pixel_y / frame_height) * pitch_width
    
    # Enforce strict pitch boundaries
    pitch_x = clamp(raw_pitch_x, 0.0, pitch_length)
    pitch_y = clamp(raw_pitch_y, 0.0, pitch_width)
    
    return round(pitch_x, 2), round(pitch_y, 2)

if __name__ == "__main__":
    # Unit Test Example
    px, py = map_pixels_to_pitch(640, 360)
    print(f"Screen Center (640, 360 px) -> Pitch Location ({px} yds, {py} yds)")

### ✏️ Phase 6 Practice Exercises

**Exercise 6.1:** Run `python tracking_utils.py`. What pitch position does pixel location $(0, 0)$ map to?

**Exercise 6.2:** If a player is detected at pixel location $(1280, 720)$, what are their mapped pitch coordinates?

## 🏆 Phase 7 & 8: Final Boss Project — Complete Application (`app.py`)

This brings all components together into a Streamlit application that handles video ingest, player tracking, heatmap rendering, and Gemini AI tactical analysis.

In [8]:
%%writefile app.py
import streamlit as st
import cv2
import pandas as pd
import numpy as np
import tempfile
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import google.generativeai as genai
from ultralytics import YOLO
from tracking_utils import map_pixels_to_pitch

# Application Setup
st.set_page_config(page_title="Soccer Vision Lab", page_icon="⚽", layout="wide")
st.title("⚽ Soccer Vision Lab: AI Tactical Analyst")
st.write("Upload match footage to track player movement, draw professional heatmaps, and generate UEFA Pro tactical insights.")

# Sidebar configuration
st.sidebar.header("⚙️ Tracking Settings")
target_player_id = st.sidebar.number_input("Target Player Track ID", min_value=1, value=1, step=1)

# Configure Gemini AI
try:
    genai.configure(api_key=st.secrets["GEMINI_API_KEY"])
    ai_model = genai.GenerativeModel("gemini-1.5-flash")
except Exception:
    st.sidebar.warning("⚠️ GEMINI_API_KEY missing in st.secrets. AI Report disabled.")
    ai_model = None

uploaded_file = st.file_uploader("Upload Match Video Clip (.mp4, .mov)", type=["mp4", "mov", "avi"])

if uploaded_file is not None:
    tfile = tempfile.NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    
    st.info("📹 Video loaded. Processing frames with YOLO Tracker...")
    
    cap = cv2.VideoCapture(tfile.name)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    yolo_model = YOLO('yolov8n.pt')
    tracking_records = []
    frame_idx = 0
    progress_bar = st.progress(0)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= 150: # Limit processing frame window for fast demo response
            break
            
        frame_idx += 1
        if total_frames > 0:
            progress_bar.progress(min(frame_idx / min(total_frames, 150), 1.0))
            
        results = yolo_model.track(frame, persist=True, verbose=False)[0]
        
        if results.boxes is not None and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.cpu().numpy()
            clss = results.boxes.cls.cpu().numpy()
            
            for box, track_id, cls in zip(boxes, track_ids, clss):
                # Class index 0 corresponds to 'person'
                if int(cls) == 0 and int(track_id) == target_player_id:
                    x1, y1, x2, y2 = box
                    center_x = (x1 + x2) / 2.0
                    center_y = y2 # Base feet position for spatial mapping
                    
                    pitch_x, pitch_y = map_pixels_to_pitch(
                        center_x, center_y, frame_width, frame_height
                    )
                    
                    tracking_records.append({
                        "frame": frame_idx,
                        "pitch_x": pitch_x,
                        "pitch_y": pitch_y
                    })
    
    cap.release()
    st.success("✅ Processing completed successfully!")
    
    if tracking_records:
        df_tracking = pd.DataFrame(tracking_records)
        
        col1, col2 = st.columns([1, 1])
        
        with col1:
            st.subheader("📍 Tactical Heatmap")
            pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
            fig, ax = pitch.draw(figsize=(8, 5))
            bin_statistic = pitch.bin_statistic(
                df_tracking['pitch_x'], df_tracking['pitch_y'], statistic='count', bins=(12, 8)
            )
            pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', alpha=0.6)
            pitch.scatter(df_tracking['pitch_x'], df_tracking['pitch_y'], c='black', s=25, ax=ax)
            st.pyplot(fig)
            
        with col2:
            st.subheader("📋 Trajectory Summary")
            avg_x = df_tracking['pitch_x'].mean()
            avg_y = df_tracking['pitch_y'].mean()
            st.metric("Avg Position X (Length)", f"{avg_x:.1f} yds")
            st.metric("Avg Position Y (Width)", f"{avg_y:.1f} yds")
            st.metric("Logged Detections", f"{len(df_tracking)} frames")
            
        st.divider()
        st.subheader("🤖 UEFA Pro AI Tactical Assessment")
        
        if ai_model and st.button("Generate AI Report"):
            prompt = f"""
            You are an elite UEFA Pro soccer analyst.
            
            Review this positional tracking log for Player Track ID #{target_player_id}:
            - Average Pitch Length (X, 0-120 yds): {avg_x:.1f}
            - Average Pitch Width (Y, 0-80 yds): {avg_y:.1f}
            - Tracking Frame Sample Count: {len(df_tracking)}
            - Range X: {df_tracking['pitch_x'].min():.1f} to {df_tracking['pitch_x'].max():.1f}
            - Range Y: {df_tracking['pitch_y'].min():.1f} to {df_tracking['pitch_y'].max():.1f}
            
            Provide a structured, professional tactical assessment containing:
            1. Key Positional Strengths
            2. Tactical Weaknesses / Structural Flaws
            3. Direct Coaching Advice for future matches
            4. World-Class Player Comparison
            """
            with st.spinner("Generating AI Tactical Report..."):
                response = ai_model.generate_content(prompt)
                st.markdown(response.text)
    else:
        st.warning(f"No tracking records found for Target Player Track ID #{target_player_id}. Try changing the ID in the sidebar.")

Writing app.py


### ✏️ Final Capstone Exercises

**Exercise 7.1:** Execute your final web application using `streamlit run app.py`.

**Exercise 7.2:** Change the target track ID in the sidebar from `1` to `2` or `3` and observe how the tactical heatmap and average metrics update in real time.

--- 

## 🏆 Mission 12 Complete
Congratulations, Coach Artin! 🏆 You have completed Mission 12 and built a full end-to-end **AI Soccer Analytics System** that converts raw video files into tactical heatmaps and automated AI coaching reports.